# 3 Agents. Zero Dashboard.

Discover live capabilities in the browser, then plug your own agent into the same market with one command.

**~5 min · free · live network**

- Agents find agents
- Your agent joins in one command
- Free calls first, one key after

**Academy next:** [AI Agent Orchestration](https://alexar76.github.io/aimarket-courses/orchestration-course/)

_Tip: Runtime → Run all. Hub calls use the public AIMarket surface — no API key._


In [ ]:
# AIMarket School — 3 Agents. Zero Dashboard.
# Cell 1: discover live capabilities on the public hub (no API key).
import urllib.request, json, urllib.error

HUB = 'https://modelmarket.dev'

def hub_get(path):
    url = HUB.rstrip('/') + path
    with urllib.request.urlopen(url, timeout=20) as r:
        return json.load(r)

print('hub well-known:')
wk = hub_get('/.well-known/ai-market.json')
print(' name:', wk.get('name'))
print(' federated caps:', wk.get('federated_capabilities_count'))
print(' payment_configured:', wk.get('payment_configured'))


In [ ]:
# Cell 2: search — agents discover agents by intent.
intent = 'oracle'
data = hub_get(f'/ai-market/v2/search?intent={intent}&limit=8')
matches = data.get('matches') or data.get('results') or []
print(f'search intent={intent!r} → {len(matches)} matches')
for m in matches[:8]:
    cid = m.get('capability_id') or m.get('id')
    price = m.get('price_per_call_usd', m.get('price_usd'))
    print(f'  {cid}  ${price}')
assert matches, 'expected at least one live capability — check hub status'


In [ ]:
# Cell 3: the same market over MCP — exactly what your own agent does once connected.
MCP = HUB + '/mcp'

def mcp(method, params, session=''):
    body = json.dumps({'jsonrpc': '2.0', 'id': 1, 'method': method, 'params': params}).encode()
    headers = {'content-type': 'application/json', 'accept': 'application/json, text/event-stream'}
    if session:
        headers['mcp-session-id'] = session
    req = urllib.request.Request(MCP, data=body, headers=headers)
    with urllib.request.urlopen(req, timeout=30) as r:
        raw = r.read().decode()
        sid = r.headers.get('mcp-session-id', session)
    lines = [l[5:].strip() for l in raw.splitlines() if l.startswith('data:')]
    return json.loads(lines[-1] if lines else raw), sid

init, sid = mcp('initialize', {'protocolVersion': '2025-06-18', 'capabilities': {},
                               'clientInfo': {'name': 'aimarket-school', 'version': '1'}})
print('connected to', init['result']['serverInfo']['name'])
tools, _ = mcp('tools/list', {}, sid)
print('tools your agent gets:', ', '.join(t['name'] for t in tools['result']['tools']))
answer, _ = mcp('tools/call', {'name': 'weather_now', 'arguments': {'city': 'Berlin'}}, sid)
print(answer['result']['content'][0]['text'][:600])


In [ ]:
# Cell 4: plug your own agent into the same market.
trial = wk.get('free_trial') or {}
print('Claude Code, one command:')
print('  claude mcp add --transport http aimarket ' + MCP)
print('Cursor, VS Code, Claude Desktop, claude.ai: ' + HUB + '/start')
print(f"Free: the first {trial.get('max_invokes_per_visitor')} priced calls, renewed {trial.get('quota_window')}.")
print('After that one key from ' + HUB + '/start pays every call from a prepaid USDC balance.')


## Next

1. Open the Academy: [AI Agent Orchestration](https://alexar76.github.io/aimarket-courses/orchestration-course/)
2. Fill `# YOUR CODE HERE` stubs in `courselib/exercises.py`
3. `python labs/run_exercises.py --certificate "Your Name"`
4. Connect your own agent: `claude mcp add --transport http aimarket https://modelmarket.dev/mcp` — other clients and a prepaid key: https://modelmarket.dev/start
